<a href="https://colab.research.google.com/github/mhmdmeri58-cyber/IP_17Pmax_Analysis/blob/main/iphone_sentiment_cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/ip_Marques.csv")
print(f"Loaded {len(df)} rows")


Mounted at /content/drive
Loaded 500 rows


In [7]:
print(df.shape)
print(df.head())

(500, 14)
              author  authorIsChannelOwner                         cid  \
0        @leoguy1609                 False  Ugzf3iYKVkH8Jf1MS1x4AaABAg   
1   @Kaushikaksh2244                 False  UgzSKwySqoDeUPOsWeh4AaABAg   
2   @mladenkisov4253                 False  UgxE5oQ9fqHGfqUwJ5Z4AaABAg   
3          @416mando                 False  UgxfCSyaAOuFMe_3t_F4AaABAg   
4  @ArnoImenaMugisha                 False  UgwvwiU6Ty4DS6TocZ54AaABAg   

                                             comment  commentsCount  \
0  What irritates me ? For the money Apple could ...           5417   
1  Can anyone tell me what app he is using for wa...           5417   
2  The desigh is the top fail here..... Ugly as i...           5417   
3  Im excited. I will be purchasing the newest ip...           5417   
4  At 9:27 can anyone tell me where to get that w...           5417   

   hasCreatorHeart                                           pageUrl  \
0            False  https://youtu.be/q0aFOxT6T

In [3]:
top_comment = df[df["type"] == "comment"].copy()
replies = df[df["type"] == "reply"].copy()

print(f"Top-level comments: {len(top_comment)}")
print(f"Replies: {len(replies)}")

Top-level comments: 443
Replies: 57


In [4]:
print(top_comment.head())

              author  authorIsChannelOwner                         cid  \
0        @leoguy1609                 False  Ugzf3iYKVkH8Jf1MS1x4AaABAg   
1   @Kaushikaksh2244                 False  UgzSKwySqoDeUPOsWeh4AaABAg   
2   @mladenkisov4253                 False  UgxE5oQ9fqHGfqUwJ5Z4AaABAg   
3          @416mando                 False  UgxfCSyaAOuFMe_3t_F4AaABAg   
4  @ArnoImenaMugisha                 False  UgwvwiU6Ty4DS6TocZ54AaABAg   

                                             comment  commentsCount  \
0  What irritates me ? For the money Apple could ...           5417   
1  Can anyone tell me what app he is using for wa...           5417   
2  The desigh is the top fail here..... Ugly as i...           5417   
3  Im excited. I will be purchasing the newest ip...           5417   
4  At 9:27 can anyone tell me where to get that w...           5417   

   hasCreatorHeart                                           pageUrl  \
0            False  https://youtu.be/q0aFOxT6TNw?si=BZGw

In [5]:
import html
import re

def clean_comment(text):
    # Ensure the comment is text
    text = str(text)

    # Convert HTML entities, such as &amp; into &
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove video timestamps, such as 5:55 or 1:05:30
    text = re.sub(
        r"\b(?:\d{1,2}:)?\d{1,2}:\d{2}\b",
        " ",
        text
    )

    # Replace line breaks and tabs with spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Replace multiple spaces with one space
    text = re.sub(r"\s+", " ", text)

    # Remove spaces from the beginning and end
    return text.strip()

analysis_df = top_comment.copy()

analysis_df["clean_text"] = (
    analysis_df["comment"].apply(clean_comment)
)

# Remove rows where the cleaned text is empty
analysis_df = analysis_df[
    analysis_df["clean_text"].ne("")
].copy()

print(
    analysis_df[
        ["comment", "clean_text"]
    ].head(10)
)

                                             comment  \
0  What irritates me ? For the money Apple could ...   
1  Can anyone tell me what app he is using for wa...   
2  The desigh is the top fail here..... Ugly as i...   
3  Im excited. I will be purchasing the newest ip...   
4  At 9:27 can anyone tell me where to get that w...   
5  The phone still looks the same, they can’t thi...   
6                                Getting this today!   
7                      The fall of apple is starting   
8  I bought the iPhone 17 pro today because my iP...   
9                    NEED Green iPhones and products   

                                          clean_text  
0  What irritates me ? For the money Apple could ...  
1  Can anyone tell me what app he is using for wa...  
2  The desigh is the top fail here..... Ugly as i...  
3  Im excited. I will be purchasing the newest ip...  
4     At can anyone tell me where to get that widget  
5  The phone still looks the same, they can’t thi... 

In [6]:
%pip install -q langdetect

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Make results reproducible
DetectorFactory.seed = 42

def detect_language(text):
    text = str(text).strip()

    # Very short comments are difficult to identify reliably
    if len(text.split()) < 3:
        return "unknown"

    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

analysis_df["language"] = (
    analysis_df["clean_text"].apply(detect_language)
)

print(analysis_df["language"].value_counts(dropna=False))

# Use English comments for the English sentiment model.
english_df = analysis_df[analysis_df["language"] == "en"].copy()
unknown_df = analysis_df[analysis_df["language"] == "unknown"].copy()
non_english_df = analysis_df[
    ~analysis_df["language"].isin(["en", "unknown"])
].copy()

print(f"English comments: {len(english_df)}")
print(f"Unknown language: {len(unknown_df)}")
print(f"Non-English comments: {len(non_english_df)}")

analysis_df.to_csv(
    "/content/drive/MyDrive/ip_Marques_cleaned_with_language.csv",
    index=False,
    encoding="utf-8-sig"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
language
en         384
unknown     33
af           3
de           3
es           3
tl           3
sv           2
id           2
sl           1
nl           1
ru           1
da           1
so           1
Name: count, dtype: int64
English comments: 384
Unknown language: 33
Non-English comments: 21


In [28]:

english_df = analysis_df[
    analysis_df["language"] == "en"
].copy()

english_df["relevance"] = ""

english_df.to_csv(
    "ip_Marques_english_for_review.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files
files.download("ip_Marques_english_for_review.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
%pip install -q transformers torch
from transformers import pipeline

sentiment_model = pipeline(
    task="sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

results = sentiment_model(
    english_df["clean_text"].tolist(),
    batch_size=16,
    truncation=True,
    max_length=512
)

english_df["sentiment"] = [
    result["label"].lower()
    for result in results
]

english_df["confidence"] = [
    result["score"]
    for result in results
]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
print(
    english_df[
        ["clean_text", "sentiment", "confidence"]
    ].head(20)
)

                                           clean_text sentiment  confidence
0   What irritates me ? For the money Apple could ...  negative    0.838705
1   Can anyone tell me what app he is using for wa...   neutral    0.934577
2   The desigh is the top fail here..... Ugly as i...  negative    0.944932
3   Im excited. I will be purchasing the newest ip...  positive    0.983343
4      At can anyone tell me where to get that widget   neutral    0.861747
5   The phone still looks the same, they can’t thi...  negative    0.812400
6                                 Getting this today!  positive    0.690258
7                       The fall of apple is starting   neutral    0.647796
8   I bought the iPhone 17 pro today because my iP...  positive    0.987543
9                     NEED Green iPhones and products  positive    0.739754
10  I bought the 17 pro max last week! loving it s...  positive    0.987280
11  I’m a music fanatic and audiophile. You’ve kil...  positive    0.873961
12  I picked

In [10]:
sentiment_summary = (
    english_df["sentiment"]
    .value_counts()
    .reindex(
        ["positive", "neutral", "negative"],
        fill_value=0
    )
    .rename_axis("sentiment")
    .reset_index(name="comment_count")
)

sentiment_summary["percentage"] = (
    sentiment_summary["comment_count"]
    / sentiment_summary["comment_count"].sum()
    * 100
).round(1)

print(sentiment_summary)

  sentiment  comment_count  percentage
0  positive            120        31.2
1   neutral            134        34.9
2  negative            130        33.9


In [11]:
english_df.to_csv(
    "/content/drive/MyDrive/ip_Marques_sentiment.csv",
    index=False,
    encoding="utf-8-sig"
)
print(
    english_df[
        english_df["confidence"] < 0.70
    ][["clean_text", "sentiment", "confidence"]]
    .head(30)
)

                                           clean_text sentiment  confidence
6                                 Getting this today!  positive    0.690258
7                       The fall of apple is starting   neutral    0.647796
12  I picked up the iPhone 17 Pro. I know people a...   neutral    0.498906
18           Iphone ultra! Of course MKBHD named it..   neutral    0.501495
20  I think the ultra should have 4-5 cameras and ...   neutral    0.497445
21                              kill the lantern fly!  negative    0.562496
22  The origins was an all aluminum body except th...   neutral    0.535230
23               getting my 17 pro tommorow in orange   neutral    0.693379
25  hmm is it time to upgrade my iphone 12 to this...  positive    0.573252
28             I'd rather wait for the new iPhone 18.   neutral    0.540880
35  I'm not an iPhone fan, but that orange is call...  positive    0.523313
37  Please can you gift me an iphone 17pro 512gb ....  negative    0.642772
39  I don’t 

In [12]:
%pip install -q vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

scores = english_df["clean_text"].apply(
    analyzer.polarity_scores
)

english_df["positive_score"] = scores.apply(
    lambda result: result["pos"]
)

english_df["neutral_score"] = scores.apply(
    lambda result: result["neu"]
)

english_df["negative_score"] = scores.apply(
    lambda result: result["neg"]
)

english_df["compound_score"] = scores.apply(
    lambda result: result["compound"]
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00


In [13]:
def assign_sentiment(compound_score):
    if compound_score >= 0.05:
        return "positive"
    elif compound_score <= -0.05:
        return "negative"
    else:
        return "neutral"

english_df["sentiment"] = (
    english_df["compound_score"].apply(assign_sentiment)
)

In [14]:
print(
    english_df[
        [
            "clean_text",
            "positive_score",
            "neutral_score",
            "negative_score",
            "compound_score",
            "sentiment"
        ]
    ].head(20)
)

                                           clean_text  positive_score  \
0   What irritates me ? For the money Apple could ...           0.084   
1   Can anyone tell me what app he is using for wa...           0.000   
2   The desigh is the top fail here..... Ugly as i...           0.151   
3   Im excited. I will be purchasing the newest ip...           0.261   
4      At can anyone tell me where to get that widget           0.000   
5   The phone still looks the same, they can’t thi...           0.000   
6                                 Getting this today!           0.000   
7                       The fall of apple is starting           0.000   
8   I bought the iPhone 17 pro today because my iP...           0.237   
9                     NEED Green iPhones and products           0.000   
10  I bought the 17 pro max last week! loving it s...           0.276   
11  I’m a music fanatic and audiophile. You’ve kil...           0.000   
12  I picked up the iPhone 17 Pro. I know people a.

In [15]:
sentiment_summary = (
    english_df["sentiment"]
    .value_counts()
    .reindex(
        ["positive", "neutral", "negative"],
        fill_value=0
    )
    .rename_axis("sentiment")
    .reset_index(name="comment_count")
)

sentiment_summary["percentage"] = (
    sentiment_summary["comment_count"]
    / sentiment_summary["comment_count"].sum()
    * 100
).round(1)

print(sentiment_summary)

  sentiment  comment_count  percentage
0  positive            193        50.3
1   neutral             93        24.2
2  negative             98        25.5


In [17]:
vader_labels = english_df["sentiment"]
roberta_labels = [
    result["label"].lower()
    for result in results
]

english_df["vader_sentiment"] = vader_labels
english_df["roberta_sentiment"] = roberta_labels

In [18]:
comparison = pd.crosstab(
    english_df["vader_sentiment"],
    english_df["roberta_sentiment"],
    margins=True
)

print(comparison)

roberta_sentiment  negative  neutral  positive  All
vader_sentiment                                    
negative                 66       19        13   98
neutral                  23       59        11   93
positive                 41       56        96  193
All                     130      134       120  384


In [19]:
agreement = (
    english_df["vader_sentiment"]
    == english_df["roberta_sentiment"]
).mean()

print(f"Model agreement: {agreement:.1%}")

Model agreement: 57.6%


In [20]:
disagreements = english_df[
    english_df["vader_sentiment"]
    != english_df["roberta_sentiment"]
][
    [
        "clean_text",
        "vader_sentiment",
        "roberta_sentiment"
    ]
]

print(disagreements.head(30))

                                           clean_text vader_sentiment  \
0   What irritates me ? For the money Apple could ...        positive   
5   The phone still looks the same, they can’t thi...         neutral   
6                                 Getting this today!         neutral   
9                     NEED Green iPhones and products         neutral   
11  I’m a music fanatic and audiophile. You’ve kil...        negative   
12  I picked up the iPhone 17 Pro. I know people a...        positive   
20  I think the ultra should have 4-5 cameras and ...        positive   
22  The origins was an all aluminum body except th...        negative   
25  hmm is it time to upgrade my iphone 12 to this...         neutral   
27                 Put a case on, you monster!!! Lol.        positive   
35  I'm not an iPhone fan, but that orange is call...        negative   
36                          That bug was a paid actor         neutral   
41                          Yeah im sticking with a

In [21]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

In [22]:
comparison_df = english_df[
    [
        "comment",
        "clean_text",
        "vader_sentiment",
        "roberta_sentiment"
    ]
]

display(comparison_df)

,comment,clean_text,vader_sentiment,roberta_sentiment
0,What irritates me ? For the money Apple could have used a better aluminum alloy for their housing and more scratch resistent coating on their iPhone 17 Pro and Pro Max.,What irritates me ? For the money Apple could have used a better aluminum alloy for their housing and more scratch resistent coating on their iPhone 17 Pro and Pro Max.,positive,negative
1,Can anyone tell me what app he is using for wallpaper at 5.56,Can anyone tell me what app he is using for wallpaper at 5.56,neutral,neutral
2,"The desigh is the top fail here..... Ugly as i ever seen...... And its a heavy like a break i my pocket. Don't worry, the heat is 200%. Oh, the battery barely make a whole day.","The desigh is the top fail here..... Ugly as i ever seen...... And its a heavy like a break i my pocket. Don't worry, the heat is 200%. Oh, the battery barely make a whole day.",negative,negative
3,Im excited. I will be purchasing the newest iphone if apple announces something great this wednesday! Haven't had iphone since the 5!! Im ready to get rid of my 2021 motorola,Im excited. I will be purchasing the newest iphone if apple announces something great this wednesday! Haven't had iphone since the 5!! Im ready to get rid of my 2021 motorola,positive,positive
4,At 9:27 can anyone tell me where to get that widget,At can anyone tell me where to get that widget,neutral,neutral
5,"The phone still looks the same, they can’t think outside the box","The phone still looks the same, they can’t think outside the box",neutral,negative
6,Getting this today!,Getting this today!,neutral,positive
7,The fall of apple is starting,The fall of apple is starting,neutral,neutral
8,"I bought the iPhone 17 pro today because my iPhone XS finally reached the end of its life, loving it!","I bought the iPhone 17 pro today because my iPhone XS finally reached the end of its life, loving it!",positive,positive
9,NEED Green iPhones and products,NEED Green iPhones and products,neutral,positive


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive
